# AECCT on LDPC(49,24)

This notebook clones the repository, checks the Kaggle GPU, trains the AECCT configuration used for comparison, and prints the resulting BER/FER log.

Before running it, push `AECCT-main/` to the GitHub repository. The current local folder is untracked, so it will not be included by `git clone` until you commit and push it. Enable **Internet** and a **GPU accelerator** in Kaggle.

In [ ]:
from pathlib import Path
import os
import subprocess
import sys

REPO_URL = 'https://github.com/gouravanirudh05/SRIP_LDPC_Decoding_using_Machine_Learning.git'
REPO_DIR = Path('/kaggle/working/ldpc_repo')

if not (REPO_DIR / '.git').exists():
    subprocess.run(['git', 'clone', REPO_URL, str(REPO_DIR)], check=True)

AECCT_DIR = REPO_DIR / 'AECCT-main'
if not (AECCT_DIR / 'main.py').exists():
    raise FileNotFoundError('AECCT-main is missing from the cloned repository. Commit and push it first.')

os.chdir(AECCT_DIR)
print('Working directory:', Path.cwd())

In [ ]:
# Kaggle normally already provides PyTorch. Install the packages required by AECCT.
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'einops', 'tqdm'], check=True)

import torch
print('PyTorch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
if not torch.cuda.is_available():
    raise RuntimeError('GPU is not available. In Kaggle, select GPU under Notebook options.')
print('GPU:', torch.cuda.get_device_name(0))

## Training configuration

AECCT uses 6 transformer blocks and dimension 128 on `LDPC_N49_K24`. `main.py` spends `--epochs` twice: phase 1 trains the float model, phase 2 reruns the same number of epochs as quantization-aware training, then converts to the inference model and evaluates.

`--epochs 700` gives 1400 training epochs total, ~8.4 h, sized to fit Kaggle's 12 h GPU session. The earlier 250-epoch run took 3.11 h and reached BER 2.33e-05 at 6 dB.

Note that even at 700 epochs this is still well short of the reference setup, which used 1000 epochs at 1000 batches per epoch (1e6 gradient steps per phase, versus 1.4e5 here). Increasing `--train_batches_per_epoch` instead of `--epochs` buys no extra gradient steps per unit of wall time, so epochs is the knob to raise: it also anneals the cosine LR schedule more smoothly, since `T_max` is set to `--epochs` and stepped once per epoch.

In [ ]:
import time

# main.py spends --epochs TWICE: once in phase 1 (float) and again in phase 2 (QAT).
# Measured on the 250-epoch run: ~13.6 s per phase-1 epoch, ~29.5 s per QAT epoch
# => ~43 s of wall time per unit of --epochs, plus ~2 x 4 min for the two evaluations.
# 700 * 43 s ~= 8.4 h, which leaves headroom inside Kaggle's 12 h GPU session.
EPOCHS = 400

# Hard stop so an overrun cannot burn the whole session. best_model is written every
# time the loss improves, so a timeout still leaves a usable checkpoint on disk.
TRAIN_BUDGET_HOURS = 10.5

command = [
    sys.executable, 'main.py',
    '--code', 'LDPC_N49_K24',
    '--N_dec', '6',
    '--d_model', '128',
    '--epochs', str(EPOCHS),
    '--workers', '4',
    '--batch_size', '128',
    '--train_batches_per_epoch', '200',
    '--test_batch_size', '2048',
    '--lr', '1e-4',
    '--seed', '42',
]
print('Running:', ' '.join(command))
print(f'Estimated training time: {EPOCHS * 43.1 / 3600:.1f} h (budget {TRAIN_BUDGET_HOURS} h)')

start = time.perf_counter()
timed_out = False
try:
    subprocess.run(command, cwd=str(AECCT_DIR), check=True,
                   timeout=TRAIN_BUDGET_HOURS * 3600)
except subprocess.TimeoutExpired:
    timed_out = True
    print(f'\n!! Training exceeded the {TRAIN_BUDGET_HOURS} h budget and was stopped.')
    print('   best_model is still on disk, but the final QAT evaluation did not run.')
    print('   Lower EPOCHS and rerun, or evaluate the checkpoint separately.')
print(f'Total wall time: {(time.perf_counter() - start) / 3600:.2f} hours')

In [ ]:
# Print the latest AECCT result directory and its final log lines.
result_dirs = sorted((AECCT_DIR / 'logs' / 'Results_AECCT').glob('*'), key=lambda p: p.stat().st_mtime)
if not result_dirs:
    raise FileNotFoundError('No AECCT result directory was produced.')
latest = result_dirs[-1]
log_file = latest / 'logging.txt'
print('Result directory:', latest)
print('Checkpoint:', latest / 'best_model')
print('\n'.join(log_file.read_text(errors='replace').splitlines()[-40:]))

In [ ]:
# Archive the latest AECCT result directory so it can be downloaded from the notebook output.
from pathlib import Path
import tarfile

result_root = Path('/kaggle/working/ldpc_repo/AECCT-main/logs/Results_AECCT')
# For an ECCT run instead, point at: Path('/kaggle/working/ldpc_repo/ECCT/Results_ECCT')

if not result_root.exists():
    raise FileNotFoundError(f'Result root does not exist: {result_root}')

result_dirs = sorted((p for p in result_root.glob('*') if p.is_dir()), key=lambda p: p.stat().st_mtime)
if not result_dirs:
    raise FileNotFoundError(f'No result directory found under {result_root}')

latest = result_dirs[-1]
archive = Path('/kaggle/working/LDPC49_AECCT_results.tar.gz')

with tarfile.open(archive, 'w:gz') as tar:
    tar.add(latest, arcname=latest.name)

print('Saved:', archive, f'({archive.stat().st_size / 1e6:.1f} MB)')
print('Log:', latest / 'logging.txt')
print('Checkpoint:', latest / 'best_model')